# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | — |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: nome del layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
I cluster che emergono rappresentano i livelli gerarchici specifici per quella materia —
quanti siano lo decide l'algoritmo, non l'analista.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro:
alcuni concentrati su livelli apicali, altri su livelli tecnici di dettaglio.
L'ibridità è misurata come **entropia media degli articoli**.

## Output

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale (LLM 1) |
| `layer_mapping.csv` | Cluster → nome layer con descrizione e rank gerarchico |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità e layer dominante |

---

> **Nota sul costo API**: Fase A chiama l'LLM una volta per segmento, Fase C una volta
> per articolo. Con ~200 atti e ~20 segmenti/atto = ordine di 4.000–6.000 chiamate.
> Il checkpointing granulare permette di riprendere da dove si era interrotti.

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
#  MATERIA  →  stessa cartella scelta nei notebook 02 e 03
# ─────────────────────────────────────────────────────────────────────────────

MATERIA_NAME = "fdi_screening"   # <- modifica qui


# ── Descrizione tema per i prompt LLM ────────────────────────────────────────
# Frase in italiano che descrive la materia. Usata nei tre prompt LLM per
# contestualizzare le descrizioni funzionali.
# Esempi:
#   "controllo degli investimenti diretti esteri (FDI Screening) e poteri
#    speciali dello Stato (Golden Power) sulla sicurezza nazionale"
#   "protezione dei dati personali e privacy nel contesto digitale europeo"

TEMA_DESCRIZIONE = (
    "Foreign direct investment screening (FDI Screening)"
    "and special state powers (Golden Power) over national security"
    "and strategic infrastructure."
)


# ── Modello OpenAI ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-5.4-mini"   # usato per tutte e tre le invocazioni LLM


# ── Parametri chiamate API ────────────────────────────────────────────────────
LLM_MAX_TOKENS_A   = 300    # Fase A: descrizioni brevi (1-2 frasi)
LLM_MAX_TOKENS_B2  = 400    # Fase B.2: JSON nome + descrizione layer
LLM_MAX_TOKENS_C   = 500    # Fase C: JSON percentuali
LLM_DELAY_SECONDS  = 0.3    # pausa tra chiamate (rispetta il rate limit)
LLM_MAX_RETRIES    = 3      # tentativi in caso di errore transitorio
LLM_RETRY_DELAY    = 5.0    # secondi tra retry


# ── Parametri checkpoint ──────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 100    # segmenti/articoli tra un salvataggio e il successivo


# ── Parametri clustering ──────────────────────────────────────────────────────
EMBEDDING_MODEL     = "all-mpnet-base-v2"
UMAP_N_COMPONENTS   = 10     # dimensioni ridotte prima di HDBSCAN
UMAP_N_NEIGHBORS    = 15
UMAP_MIN_DIST       = 0.0    # 0.0 ottimizza la separazione dei cluster
HDBSCAN_MIN_CLUSTER = 5      # minimo segmenti per formare un layer
HDBSCAN_MIN_SAMPLES = 3
N_REPR_DESCRIPTIONS = 10     # descrizioni representative per il naming

## 1. Import e Percorsi

In [19]:
import os
import json
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts.csv')
EDGES_FILE              = os.path.join(output_path, 'edges_focal.csv')
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, 'layer_mapping.csv')
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CKPT_FILE       = os.path.join(output_path, 'heatmap_checkpoint.csv')

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {INPUT_FILE}")
print(f"Modello LLM:   {LLM_MODEL}")
print(f"Embedding:     {EMBEDDING_MODEL}")

Materia:       fdi_screening
Input:         ..\data\output\fdi_screening\nodes_texts.csv
Modello LLM:   gpt-5.4-mini
Embedding:     all-mpnet-base-v2


## 2. Caricamento Dati

In [3]:
nodes = pd.read_csv(INPUT_FILE)
print(f"Nodi totali: {len(nodes)}")

REQUIRED_COLS = ['Id', 'Label', 'segments', 'text_status', 'title']
missing_cols = [c for c in REQUIRED_COLS if c not in nodes.columns]
if missing_cols:
    raise RuntimeError(
        f"Colonne mancanti in {INPUT_FILE}: {missing_cols}.\n"
        "Assicurarsi che il notebook 03 sia stato eseguito completamente."
    )

nodes_ok   = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()

print(f"Atti con testo (text_status=ok): {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):      {len(nodes_fail)}")
print()
print("Distribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

Nodi totali: 23
Atti con testo (text_status=ok): 23
Atti senza testo (esclusi):      0

Distribuzione text_status:
text_status
ok    23


## 3. Esplosione dei Segmenti

La colonna `segments` di ogni atto contiene una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con **una riga per segmento** — l'unità di analisi del clustering.

In [4]:
def parse_segments(row):
    """Parsa la colonna 'segments' e restituisce lista di dict arricchiti."""
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':          celex,
            'node_id':        row['Id'],
            'title_atto':     title,
            'tipo':           s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':          s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove segmenti con testo troppo breve per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].reset_index(drop=True)

print(f"Segmenti estratti:              {before:,}")
print(f"Segmenti validi (>={MIN_TESTO_LEN} car): {len(segments_df):,}")
print(f"Segmenti scartati:              {before - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print(f"Segmenti medi per atto:  {segments_df.groupby('celex').size().mean():.1f}")

Segmenti estratti:              2,608
Segmenti validi (>=30 car): 2,593
Segmenti scartati:              15

Distribuzione per tipo:
tipo
articolo            1955
considerando         615
preambolo_header      23

Segmenti medi per atto:  136.5


## 4. Fase A — Descrizione Funzionale per Segmento (LLM 1)

Per ogni segmento l'LLM produce una **descrizione funzionale contestualizzata alla materia**:
cosa fa normativamente quel segmento in questo specifico contesto.

Questo è il passaggio che rende il metodo specifico per materia: lo stesso articolo
"Definizioni" viene descritto diversamente in un regolamento FDI rispetto a uno sulla privacy.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv` ogni
> `CHECKPOINT_EVERY` segmenti. Rieseguire la cella riprende dal punto di interruzione.

In [5]:
def build_prompt_functional_description(testo, tipo, identificatore, title_atto, tema):
    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Leggi questo segmento di un atto normativo e descrivi in 1-2 frasi cosa fa
normativamente — quale funzione specifica svolge in questa materia.

Sii preciso sulla funzione. Esempi di descrizioni corrette:
- "Stabilisce i principi fondamentali che giustificano l'intervento statale negli investimenti esteri per ragioni di sicurezza nazionale."
- "Definisce la procedura di notifica preventiva che gli investitori esteri devono seguire prima di acquisire partecipazioni in settori strategici."
- "Specifica le soglie percentuali di controllo societario oltre le quali scatta l'obbligo di screening."
- "Elenca i settori economici considerati critici ai fini dell'applicazione dei poteri di intervento statale."

Titolo atto: {title_atto}
Segmento ({tipo} {identificatore}):
{testo}

Rispondi SOLO con la descrizione funzionale (1-2 frasi), nessun altro testo."""


def call_llm(client, prompt, max_tokens):
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.
    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(LLM_MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=max_tokens,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (tentativo {attempt+1}/{LLM_MAX_RETRIES})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni LLM definite.")

Funzioni LLM definite.


In [10]:
# ── Gestione checkpoint ───────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint — si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si può passare alla Fase B.")

Nessun checkpoint — si parte da zero.
Da descrivere: 2,593


In [29]:
%% time
client = OpenAI()   # legge OPENAI_API_KEY dall'environment

new_rows = []
n_ok = n_error = 0
total = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)

    new_rows.append({
        'segment_id':              seg['segment_id'],
        'celex':                   seg['celex'],
        'node_id':                 seg['node_id'],
        'tipo':                    seg['tipo'],
        'identificatore':          seg['identificatore'],
        'testo_originale':         seg['testo'],
        'descrizione_funzionale':  descrizione,
        'llm_status':              status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([segs_done, batch], ignore_index=True) if not segs_done.empty else batch
        print(f"  [{i+1:>5}/{total}]  {(i+1)/total*100:5.1f}%   ok: {n_ok}   errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE A — ok: {n_ok:,}   errori: {n_error:,}")
print("=" * 50)

UsageError: Cell magic `%%` not found.


## 5. Fase B — Embedding e Clustering → Layer Emergenti

Le descrizioni funzionali vengono embeddate con `all-mpnet-base-v2`, ridotte con
UMAP e clusterizzate con HDBSCAN. Ogni cluster che emerge è un **livello gerarchico
specifico per questa materia** — quanti ce ne sono lo decide l'algoritmo.

I segmenti assegnati al cluster `-1` (noise) vengono conservati ma esclusi dal naming.

In [ ]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Carica dal checkpoint finale
segs_desc = pd.read_csv(SEGMENTS_DESC_FILE)

segs_valid = segs_desc[
    (segs_desc['llm_status'] == 'ok') &
    segs_desc['descrizione_funzionale'].notna() &
    (segs_desc['descrizione_funzionale'].str.len() > 10)
].copy()

print(f"Segmenti con descrizione valida: {len(segs_valid):,} / {len(segs_desc):,}")
print()

print(f"Caricamento modello embedding: {EMBEDDING_MODEL} ...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

print("Calcolo embeddings...")
embeddings = encoder.encode(
    segs_valid['descrizione_funzionale'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
print(f"UMAP: {embeddings.shape[1]}d → {UMAP_N_COMPONENTS}d ...")

reducer = umap.UMAP(
    n_components = UMAP_N_COMPONENTS,
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    metric       = 'cosine',
    random_state = 42,
    low_memory   = False,
)
embeddings_reduced = reducer.fit_transform(embeddings)
print(f"Shape ridotta: {embeddings_reduced.shape}")

In [ ]:
print(f"HDBSCAN (min_cluster={HDBSCAN_MIN_CLUSTER}, min_samples={HDBSCAN_MIN_SAMPLES}) ...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size         = HDBSCAN_MIN_CLUSTER,
    min_samples              = HDBSCAN_MIN_SAMPLES,
    cluster_selection_method = 'eom',
    prediction_data          = True,
)
cluster_labels = clusterer.fit_predict(embeddings_reduced)

segs_valid = segs_valid.copy()
segs_valid['cluster_id'] = cluster_labels
if hasattr(clusterer, 'probabilities_'):
    segs_valid['cluster_prob'] = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print()
print("=" * 50)
print("RISULTATO CLUSTERING")
print("=" * 50)
print(f"  Layer trovati:          {n_clusters}")
print(f"  Segmenti noise (-1):    {n_noise:,}  ({n_noise/len(cluster_labels)*100:.1f}%)")
print()

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("Distribuzione segmenti per cluster:")
for cid, cnt in cluster_counts.items():
    tag = 'NOISE' if cid == -1 else f'cluster_{cid}'
    bar = '█' * min(40, int(cnt / cluster_counts.max() * 40))
    print(f"  {tag:>12}: {cnt:>5}  {bar}")

## 6. Fase B.2 — Ordinamento Gerarchico e Naming dei Layer (LLM 2)

Due operazioni:

1. **Ordinamento gerarchico**: i cluster vengono ordinati per posizione nella gerarchia
   usando l'indegree della rete di citazioni come proxy.
   Un atto molto citato (alto indegree) è probabilmente fondativo → contiene segmenti apicali.

2. **Naming**: LLM 2 riceve le 10 descrizioni più rappresentative di ogni cluster
   e assegna un nome e una descrizione al layer.

In [ ]:
# ── Ordinamento gerarchico via indegree ───────────────────────────────────────
try:
    edges_df = pd.read_csv(EDGES_FILE)
    # edges_focal.csv ha colonne Source, Target (ID numerici dei nodi)
    indegree = edges_df['Target'].value_counts().rename('indegree')

    # Mappa node_id → indegree
    segs_with_deg = segs_valid.copy()
    segs_with_deg['indegree'] = segs_with_deg['node_id'].map(indegree).fillna(0)

    # Per ogni cluster: indegree medio dei propri segmenti
    # cluster con indegree medio alto → contiene norme apicali → rank basso (=1)
    cluster_hierarchy = (
        segs_with_deg[segs_with_deg['cluster_id'] != -1]
        .groupby('cluster_id')['indegree']
        .mean()
        .sort_values(ascending=False)
    )
    cluster_rank = {cid: rank for rank, cid in enumerate(cluster_hierarchy.index, start=1)}
    print(f"Archi caricati: {len(edges_df):,}")
    print("Ordinamento gerarchico (indegree medio per cluster):")
    for cid, mean_ind in cluster_hierarchy.items():
        print(f"  cluster_{cid} → rank {cluster_rank[cid]}  (indegree medio: {mean_ind:.2f})")

except FileNotFoundError:
    print(f"⚠️  {EDGES_FILE} non trovato — ordinamento per dimensione cluster.")
    non_noise = cluster_counts[cluster_counts.index != -1].sort_values(ascending=False)
    cluster_rank = {cid: rank for rank, cid in enumerate(non_noise.index, start=1)}

print(f"\nRanking finale: {cluster_rank}")

In [ ]:
def get_representative_descriptions(segs_df, cluster_id, n=N_REPR_DESCRIPTIONS):
    """N descrizioni più rappresentative: per probabilità HDBSCAN, altrimenti casuale."""
    subset = segs_df[segs_df['cluster_id'] == cluster_id].copy()
    if 'cluster_prob' in subset.columns:
        subset = subset.nlargest(n, 'cluster_prob')
    else:
        subset = subset.sample(min(n, len(subset)), random_state=42)
    return subset['descrizione_funzionale'].tolist()


def build_prompt_layer_naming(descriptions, tema, rank, n_total):
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions)])
    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Ho condotto un'analisi clustering su un corpus di atti normativi e ho trovato
{n_total} livelli gerarchici distinti presenti in questa materia.
Questo è il livello {rank} su {n_total} in ordine dalla norma più apicale
alla più tecnica di dettaglio.

Ecco le descrizioni funzionali dei segmenti più rappresentativi di questo livello:

{desc_list}

Assegna un nome preciso a questo livello gerarchico e scrivi una breve descrizione
(2-3 frasi) di cosa caratterizza le norme che vi appartengono.

Rispondi SOLO in questo formato JSON (nessun altro testo, nessun backtick):
{{"nome": "Nome del Layer", "descrizione": "Descrizione in 2-3 frasi."}}"""


n_valid_clusters = len([cid for cid in set(cluster_labels) if cid != -1])
layer_records    = []

for cluster_id in sorted(cluster_rank.keys()):
    rank       = cluster_rank[cluster_id]
    repr_descs = get_representative_descriptions(segs_valid, cluster_id)

    prompt = build_prompt_layer_naming(
        descriptions = repr_descs,
        tema         = TEMA_DESCRIZIONE,
        rank         = rank,
        n_total      = n_valid_clusters,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_B2)

    nome        = f'Layer_{rank}'
    descrizione = ''
    if status == 'ok':
        try:
            clean       = response_text.replace('```json', '').replace('```', '').strip()
            parsed      = json.loads(clean)
            nome        = parsed.get('nome', nome)
            descrizione = parsed.get('descrizione', '')
        except json.JSONDecodeError:
            nome = response_text[:80].strip()
            print(f"  ⚠️  cluster_{cluster_id}: JSON non valido, uso raw text come nome")

    layer_records.append({
        'cluster_id':        cluster_id,
        'layer_rank':        rank,
        'layer_name':        nome,
        'layer_description': descrizione,
        'n_segments':        int((segs_valid['cluster_id'] == cluster_id).sum()),
        'repr_descriptions': json.dumps(repr_descs, ensure_ascii=False),
        'llm_status':        status,
    })
    print(f"  Rank {rank:>2} | cluster_{cluster_id:>3} → '{nome}'")
    time.sleep(LLM_DELAY_SECONDS)

layer_mapping_df = pd.DataFrame(layer_records).sort_values('layer_rank')
layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)

print()
print("=" * 60)
print("LAYER TROVATI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {str(row['layer_description'])[:120]}")
    print(f"      N segmenti: {row['n_segments']:,}")
    print()

## 7. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer noti, per ogni **articolo** di ogni atto l'LLM produce una distribuzione
percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è la matrice **articoli × layer** (valori = %) che alimenta
la heatmap nell'applicazione — identica concettualmente alla figura 12 del paper KG-Codex,
ma con layer specifici per questa materia invece che predefiniti.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica:
> spesso coprono più livelli intenzionalmente per costruire l'argomentazione legale.
> La varianza dei considerando riflette struttura retorica, non patologia.
> Gli articoli hanno funzione prescrittiva — la loro ibridità è il segnale diagnostico.

In [ ]:
# Ricarica layer mapping (può essere eseguita anche senza rieseguire B)
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
layer_list = [
    {
        'rank':        int(row['layer_rank']),
        'name':        row['layer_name'],
        'description': row['layer_description'],
    }
    for _, row in layer_mapping_df.sort_values('layer_rank').iterrows()
]
layer_names = [l['name'] for l in layer_list]

# Colonne CSV safe (senza spazi/slash)
def to_col(name):
    return 'pct__' + name.replace(' ', '_').replace('/', '_')[:50]

pct_cols     = [to_col(n) for n in layer_names]
col_to_layer = {to_col(n): n for n in layer_names}

# Solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

print(f"Layer trovati: {len(layer_list)}")
for l in layer_list:
    print(f"  [{l['rank']}] {l['name']}")
print()
print(f"Articoli da classificare: {len(articles_df):,}")
print(f"Atti coinvolti:           {articles_df['celex'].nunique():,}")

In [ ]:
def build_prompt_percentage_distribution(testo, identificatore, layer_list, tema):
    layers_desc = '\n'.join([
        f"  {l['rank']}. {l['name']}: {l['description']}"
        for l in layer_list
    ])
    layer_keys = ', '.join([f'"{l["name"]}"' for l in layer_list])

    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Leggi questo articolo e distribuisci il suo contenuto in percentuale tra i seguenti
livelli normativi emersi dall'analisi di questa materia:

{layers_desc}

Regole:
- Le percentuali devono sommare esattamente a 100.
- Assegna 0 ai livelli non presenti nell'articolo.
- Se un articolo mescola livelli, rifletti le proporzioni reali.

Articolo {identificatore}:
{testo}

Rispondi SOLO con JSON valido (nessun altro testo, nessun backtick):
{{{layer_keys}: <percentuale intera>}}"""


def parse_percentage_response(response_text, layer_names):
    """
    Parsa la risposta JSON e normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean  = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    values = {name: float(parsed.get(name, 0)) for name in layer_names}
    total  = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:   # normalizza se la somma si discosta
        values = {k: v / total * 100 for k, v in values.items()}
    return values


print("Funzioni Fase C definite.")

In [ ]:
%%time
# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(HEATMAP_CKPT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CKPT_FILE)
    done_seg_ids = set(heatmap_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_seg_ids):,} articoli già classificati.")
else:
    heatmap_done = pd.DataFrame()
    done_seg_ids = set()
    print("Nessun checkpoint heatmap — si parte da zero.")

articles_todo = articles_df[~articles_df['segment_id'].isin(done_seg_ids)].copy()
print(f"Articoli da classificare: {len(articles_todo):,}")
print()

new_rows = []
n_ok_c = n_error_c = 0
total_c = len(articles_todo)

for i, (_, art) in enumerate(articles_todo.iterrows()):

    prompt = build_prompt_percentage_distribution(
        testo          = art['testo'],
        identificatore = art['identificatore'],
        layer_list     = layer_list,
        tema           = TEMA_DESCRIZIONE,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_C)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':  art['segment_id'],
        'celex':       art['celex'],
        'node_id':     art['node_id'],
        'articolo_id': art['identificatore'],
        'llm_status':  status,
    }
    for col, name in zip(pct_cols, layer_names):
        row[col] = round(distribution[name], 2) if distribution else 0.0

    if distribution:
        n_ok_c += 1
    else:
        n_error_c += 1

    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_c:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([heatmap_done, batch], ignore_index=True) if not heatmap_done.empty else batch
        combined.to_csv(HEATMAP_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_c}]  {(i+1)/total_c*100:5.1f}%   ok: {n_ok_c}   errori: {n_error_c}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE C — ok: {n_ok_c:,}   errori: {n_error_c:,}")
print("=" * 50)

In [ ]:
# Salva heatmap finale
heatmap_final = pd.read_csv(HEATMAP_CKPT_FILE)
heatmap_final.to_csv(NODES_HEATMAP_FILE, index=False)

print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}  |  Colonne pct: {pct_cols}")
print()

ok_mask = heatmap_final['llm_status'] == 'ok'
print("Distribuzione media % per layer (articoli ok):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    bar = '█' * int(mean_pct / 2)
    print(f"  {col_to_layer[col][:45]:.<46} {mean_pct:5.1f}%  {bar}")

## 8. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica: entropia di Shannon normalizzata per articolo**

$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)$$

Lo score dell'atto è la **media delle entropie dei propri articoli**.
Zero = tutti gli articoli sono monofunzionali. Uno = distribuzione uniforme su tutti i layer.

In [ ]:
def entropy_norm(row, pct_cols):
    """Entropia di Shannon normalizzata (0=puro, 1=uniforme)."""
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    raw   = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0
    return float(raw / maxH)


def dominant_layer(row, pct_cols, col_to_layer):
    best = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(best, best)


# Solo articoli classificati correttamente
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()
heatmap_ok['entropy']       = heatmap_ok.apply(lambda r: entropy_norm(r, pct_cols), axis=1)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1)

# ── Aggregazione per atto ──────────────────────────────────────────────────────
def agg_atto(group):
    return pd.Series({
        'hybridity_score':    group['entropy'].mean(),
        'hybridity_std':      group['entropy'].std(),
        'hybridity_max':      group['entropy'].max(),
        'n_articles':         len(group),
        'dominant_layer':     group['dominant_layer'].mode().iloc[0] if len(group) > 0 else '',
        'dominant_layer_pct': (group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
                               if len(group) > 0 else 0.0),
        'most_hybrid_article': (group.nlargest(1, 'entropy')['articolo_id'].iloc[0]
                                if len(group) > 0 else ''),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols = [c for c in ['Id', 'Label', 'title', 'LegalType', 'Year', 'PipelineLevel']
             if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
nodes_meta = nodes_meta.rename(columns={'Label': 'celex'}) if 'Label' in nodes_meta.columns else nodes_meta

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)
hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
desc = hybridity_df['hybridity_score'].describe()
print("Statistiche score ibridità:")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")

## 9. Diagnostica e Verifica Qualità

In [ ]:
print("=" * 60)
print("LAYER EMERSI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    n    = row['n_segments']
    pct  = n / len(segs_valid) * 100
    bar  = '█' * int(pct)
    print(f"[{row['layer_rank']:>2}] {row['layer_name']}")
    print(f"     Segmenti: {n:,}  ({pct:.1f}%)  {bar}")
    print(f"     {str(row['layer_description'])[:110]}")
    print()

In [ ]:
print("=" * 60)
print("TOP 10 ATTI PIÙ IBRIDI")
print("=" * 60)
print()
for _, row in hybridity_df.head(10).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {row['hybridity_score']:.4f}  {celex_label}")
    print(f"           Layer dom: {row['dominant_layer']} ({row['dominant_layer_pct']:.0f}% art.)")
    print(f"           Art: {row['n_articles']}  |  Più ibrido: {row['most_hybrid_article']}")
    print(f"           {str(row.get('title',''))[:70]}")
    print()

In [ ]:
# Heatmap testuale dell'atto più ibrido
top_celex = hybridity_df.iloc[0]['celex']
top_score = hybridity_df.iloc[0]['hybridity_score']

print("=" * 60)
print(f"HEATMAP — atto più ibrido: {top_celex}  (score: {top_score:.4f})")
print("=" * 60)

act_df = (heatmap_ok[heatmap_ok['celex'] == top_celex]
          [['articolo_id'] + pct_cols + ['entropy']]
          .sort_values('entropy', ascending=False)
          .copy())

# Rinomina colonne per display
short_names = {col: col_to_layer[col][:20] for col in pct_cols}
act_display = act_df.rename(columns=short_names)
layer_disp  = list(short_names.values())
for c in layer_disp:
    act_display[c] = act_display[c].apply(lambda x: f"{x:.0f}%")
act_display['H'] = act_df['entropy'].apply(lambda x: f"{x:.3f}")

with pd.option_context('display.max_columns', None, 'display.width', 200, 'display.max_rows', 50):
    print(act_display[['articolo_id'] + layer_disp + ['H']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma ibridità
ax1 = axes[0]
ax1.hist(hybridity_df['hybridity_score'], bins=20, edgecolor='black', color='steelblue')
ax1.axvline(hybridity_df['hybridity_score'].median(), color='red', linestyle='--',
            label=f"Mediana = {hybridity_df['hybridity_score'].median():.3f}")
ax1.set_xlabel('Hybridity Score')
ax1.set_ylabel('N atti')
ax1.set_title('Distribuzione Score di Ibridità')
ax1.legend()

# Top 15 atti
ax2 = axes[1]
top15  = hybridity_df.head(15)
labels = top15['celex'].apply(lambda x: str(x)[:14]).tolist()
ax2.barh(range(len(top15)), top15['hybridity_score'], color='tomato')
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels(labels, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Hybridity Score')
ax2.set_title('Top 15 Atti più Ibridi')

plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'hybridity_distribution.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {fig_path}")

## 10. Riepilogo Output

Verifica che tutti i file di output siano stati prodotti correttamente.

In [ ]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Pipeline 04 completata.")
    print("  → Il notebook 05_network_analysis.ipynb può ora essere eseguito.")
else:
    print("⚠️  Alcuni file mancano — rieseguire le celle corrispondenti.")